# Subred a Trabajar 


## 1. Datos

El notebook se arma solo: define la subred y carga septiembre desde `data/raw/` (primero hay que correr `data_cleaning.ipynb` para que los CSV estén descargados).

Por ahora lee los datos **crudos**. Cuando `data_cleaning.ipynb` guarde los datos limpios, esta celda se cambia para leer ese archivo.

Columnas que usa el mapa:
- `fecha`, `hora_retiro`: día y hora del retiro.
- `es_habil`: lunes a viernes, sin el festivo del 16 de septiembre ni los días irregulares del 14 y 15 (cierre de la zona Buenavista).
- `retiro_subred`, `arribo_subred`: si el retiro o el arribo es una de las 10 estaciones.

In [ ]:
from pathlib import Path
import json

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import contextily as ctx
from matplotlib.patches import FancyArrowPatch
from matplotlib.lines import Line2D

RUTA_VIAJES = Path("data/raw/trips/2026-09.csv")
RUTA_ESTACIONES = Path("data/raw/station_information.json")
RUTA_FIGURA = Path("outputs/figuras/red_subred_am.png")

SUBRED = {
    "271-272": "Buenavista",
    "273-274": "Buenavista",
    "266-267": "Buenavista",
    "268-269": "Buenavista",
    "014": "Reforma",
    "474": "Reforma",
    "027": "Reforma",
    "018": "Reforma",
    "017": "Reforma",
    "015": "Reforma",
}

FESTIVOS = pd.to_datetime(["2026-09-16"]).date
DIAS_IRREGULARES = pd.to_datetime(["2026-09-14", "2026-09-15"]).date

df = pd.read_csv(RUTA_VIAJES, dtype=str)
retiro = pd.to_datetime(df["Fecha_Retiro"] + " " + df["Hora_Retiro"], format="%d/%m/%Y %H:%M:%S")
df = df[retiro.dt.strftime("%Y-%m") == "2026-09"].copy()   # solo retiros de septiembre
retiro = retiro[df.index]

df["fecha"] = retiro.dt.date
df["hora_retiro"] = retiro.dt.hour
df["es_habil"] = (retiro.dt.day_of_week < 5) & ~df["fecha"].isin(FESTIVOS) & ~df["fecha"].isin(DIAS_IRREGULARES)
df["retiro_subred"] = df["Ciclo_Estacion_Retiro"].isin(SUBRED)
df["arribo_subred"] = df["Ciclo_EstacionArribo"].isin(SUBRED)

print(f"Viajes con retiro en septiembre: {len(df):,}")
print(f"Días hábiles: {df.loc[df.es_habil, 'fecha'].nunique()}")

## 2. Mapa de flujos

In [ ]:
# --- 1. Coordenadas de las 10 estaciones (Web Mercator, para el mapa de fondo)
info = pd.DataFrame(json.loads(RUTA_ESTACIONES.read_text())["data"]["stations"])
est = info[info.short_name.isin(SUBRED)].set_index("short_name")
R = 6378137
est["x"] = R * np.radians(est.lon)
est["y"] = R * np.log(np.tan(np.pi/4 + np.radians(est.lat)/2))
est["grupo"] = est.index.map(SUBRED)

# --- 2. Flujos origen-destino dentro de la subred (6–10 am, días hábiles)
m = (df.es_habil & df.retiro_subred & df.arribo_subred
     & df.hora_retiro.between(6, 9)
     & (df.Ciclo_Estacion_Retiro != df.Ciclo_EstacionArribo))
n_dias = df.loc[df.es_habil, "fecha"].nunique()
od = (df[m].groupby(["Ciclo_Estacion_Retiro", "Ciclo_EstacionArribo"])
           .size().div(n_dias).rename("viajes_dia").reset_index())
od = od[od.viajes_dia >= 1]          # solo flujos de al menos 1 viaje/día

# --- 3. Dibujo
colores = {"Buenavista": "#d1495b", "Reforma": "#2e86ab"}
fig, ax = plt.subplots(figsize=(10, 9))

for _, r in od.iterrows():
    o, d = est.loc[r.Ciclo_Estacion_Retiro], est.loc[r.Ciclo_EstacionArribo]
    ax.add_patch(FancyArrowPatch(
        (o.x, o.y), (d.x, d.y),
        connectionstyle="arc3,rad=0.15",          # curva leve para que no se encimen
        arrowstyle="-|>", mutation_scale=12,
        lw=0.5 + r.viajes_dia / 2,                 # grosor ∝ viajes por día
        color=colores[o.grupo], alpha=0.6, zorder=2))

ax.scatter(est.x, est.y, s=est.capacity * 4,
           c=est.grupo.map(colores), edgecolor="white", lw=1.5, zorder=3)
for code, r in est.iterrows():
    ax.annotate(code, (r.x, r.y), xytext=(6, 6), textcoords="offset points",
                fontsize=9, weight="bold", zorder=4)

pad = 400                                           # margen en metros
ax.set_xlim(est.x.min() - pad, est.x.max() + pad)
ax.set_ylim(est.y.min() - pad, est.y.max() + pad)
ctx.add_basemap(ax, crs="EPSG:3857", source=ctx.providers.Esri.WorldGrayCanvas)   # CartoDB ya pide API key
ax.set_axis_off()
ax.set_title("Flujos dentro de la subred, 6–10 am (viajes por día hábil)")

ax.legend(handles=[Line2D([], [], marker="o", ls="", color=c, label=g, markersize=9)
                   for g, c in colores.items()], loc="lower right")

RUTA_FIGURA.parent.mkdir(parents=True, exist_ok=True)
plt.savefig(RUTA_FIGURA, dpi=200, bbox_inches="tight")
plt.show()